# Glioma SS BENCH Pipeline

--------

In [1]:
import numpy as np
import matplotlib as plt
import numba
import scipy
import subprocess
import random
import os
from pathlib import Path

from bench import diffusion_models
from bench.diffusion_models import sample_signal, standard_model
from fsl.data.image import Image
from fsl.wrappers.fslmaths import fslmaths
from fsl.wrappers import bet, flirt, fnirt, applywarp
from fsl.utils.image.roi import roi

In [2]:
os.environ["FSLDIR"] = "/Users/ecd25/fsl"
os.environ["FSLOUTPUTTYPE"] = "NIFTI_GZ"
os.environ["PATH"] = "/Users/ecd25/fsl/bin:" + os.environ["PATH"]

In [3]:
MNI = "/Users/ecd25/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz"
MNI_HEAD = "/Users/ecd25/fsl/data/standard/MNI152_T1_2mm.nii.gz"
CONFIG = "/Users/ecd25/fsl/etc/flirtsch/T1_2_MNI152_2mm.cnf"

In [4]:
input = "/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/input/raw"
output = "/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives"

In [107]:
# T1
T1 = f"{input}/sub-15/anat/sub-15_ses-01_T1w.nii.gz"
T1_brain = f"{output}/sub-15/anat/sub-15_ses-01_T1w_brain.nii.gz"
WM_in_MNI = "/Users/ecd25/Desktop/Fall_26/BENCH_4_WMSG/WM_in_MNI.nii.gz"
WM_in_MNI_bin = "/Users/ecd25/Desktop/Fall_26/BENCH_4_WMSG/WM_in_MNI_bin.nii.gz"

In [90]:
# Diffusion
FULL_DATA = f"{input}/sub-15/dwi/sub-15_ses-01_dwi.nii.gz"
bvecs = "/Users/ecd25/Desktop/Fall_26/protocols/on_1600_bvecs"
bvals = "/Users/ecd25/Desktop/Fall_26/protocols/on_1600.bval"

DATA_s = f"{output}/sub-15/dwi/b1600.nii.gz"
DATA = f"{output}/sub-15/dwi/b01600.nii.gz"
B0_all = f"{output}/sub-15/dwi/b0_all.nii.gz" 
B0 = f"{output}/sub-15/dwi/b0_av.nii.gz"
B0_1 = f"{output}/sub-15/dwi/b0_1.nii.gz"
B0_2 = f"{output}/sub-15/dwi/b0_2.nii.gz"
B0_3 = f"{output}/sub-15/dwi/b0_3.nii.gz"
B0_4 = f"{output}/sub-15/dwi/b0_4.nii.gz"

SUM = "/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/summaries/sub-15"

In [91]:
# Transforms
NONLIN = f"{output}/sub-15/xfms/sub-15_T1_in_MNI_nonlin.nii.gz"
OMAT = f"{output}/sub-15/xfms/sub-15_T1_2_MNI.mat"
COUT = f"{output}/sub-15/xfms/sub-15_T1_2_MNI_warp.nii.gz"
T1_in_MNI = f"{output}/sub-15/xfms/sub-15_T1_in_MNI.nii.gz"
t1_2_mni_warp = f"{output}/sub-15/xfms/sub-15_T1_2_MNI_warp.nii.gz"
DIFF2STR = f"{output}/sub-15/xfms/diff2str"
DIFF2STD = f"{output}/sub-15/xfms/diff2standard_warp.nii.gz"
B0_in_MNI = f"{output}/sub-15/xfms/sub-15_b0_in_MNI.nii.gz"

In [10]:
def _run(cmd):
    cmd = [str(c) for c in cmd]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

In [92]:
def betT1(T1, T1_brain):
    """
    Brain extraction using FSL's BET tool.
    """
    print(f"Running BET on {T1}")
    _run([
        "bet", T1, T1_brain
        ])
    
    return T1_brain
    print(f"Brain extracted image saved to {T1_brain}")

In [93]:
betT1(T1, T1_brain)

Running BET on /Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/input/raw/sub-15/anat/sub-15_ses-01_T1w.nii.gz
bet /Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/input/raw/sub-15/anat/sub-15_ses-01_T1w.nii.gz /Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/anat/sub-15_ses-01_T1w_brain.nii.gz


'/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/anat/sub-15_ses-01_T1w_brain.nii.gz'

In [94]:
def run_initial_affine(t1_brain, mni_brain, omat, out):
    """Step 1: FLIRT 12-dof affine, skull-stripped T1 -> MNI brain."""
    Path(omat).parent.mkdir(parents=True, exist_ok=True)
    _run([
        "flirt",
        "-in", t1_brain,
        "-ref", mni_brain,
        "-omat", omat,
        "-out", out,
        "-dof", 12,
        "-cost", "corratio",
        "-searchrx", -90, 90,
        "-searchry", -90, 90,
        "-searchrz", -90, 90,
    ])
    return omat

In [95]:
def run_nonlinear_registration(t1_head, mni_head, omat, config, cout, nonlin):
    """Step 2: FNIRT, whole-head T1 -> whole-head MNI, initialised with the affine."""
    _run([
        "fnirt",
        f"--in={t1_head}",
        f"--ref={mni_head}",
        f"--aff={omat}",
        f"--config={config}",
        f"--cout={cout}",
        f"--iout={nonlin}",
        "--verbose",
    ])
    return cout

In [96]:
def apply_warp(img, mni_ref, warp, out, interp="trilinear", binarize_thr=None):
    """
    Step 3: applywarp for an image in T1 space -> MNI space.
    For probabilistic maps (FAST pve), use trilinear; optionally binarize with binarize_thr.
    """
    _run([
        "applywarp",
        f"--in={img}",
        f"--ref={mni_ref}",
        f"--warp={warp}",
        f"--out={out}",
        f"--interp={interp}",
    ])
    if binarize_thr is not None:
        _run(["fslmaths", out, "-thr", binarize_thr, "-bin", out])
    return out

if __name__ == "__main__":
    run_initial_affine(T1_brain, MNI, OMAT, T1_in_MNI)
    run_nonlinear_registration(T1, MNI_HEAD, OMAT, CONFIG, COUT, NONLIN)

flirt -in /Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/anat/sub-15_ses-01_T1w_brain.nii.gz -ref /Users/ecd25/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz -omat /Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/xfms/sub-15_T1_2_MNI.mat -out /Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/xfms/sub-15_T1_in_MNI.nii.gz -dof 12 -cost corratio -searchrx -90 90 -searchry -90 90 -searchrz -90 90
fnirt --in=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/input/raw/sub-15/anat/sub-15_ses-01_T1w.nii.gz --ref=/Users/ecd25/fsl/data/standard/MNI152_T1_2mm.nii.gz --aff=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/xfms/sub-15_T1_2_MNI.mat --config=/Users/ecd25/fsl/etc/flirtsch/T1_2_MNI152_2mm.cnf --cout=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/xfms/sub-15_T1_2_MNI_warp.nii.gz --iout=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one

In [97]:
!cat $bvals

1600  1600 1600 1600 1600 1600 1600 1600 0 0.1 0.01 0.001


In [98]:
!fslroi $FULL_DATA $DATA_s 18 8

In [99]:
!fslroi $FULL_DATA $B0_1 0 1
!fslroi $FULL_DATA $B0_2 17 1
!fslroi $FULL_DATA $B0_3 26 1
!fslroi $FULL_DATA $B0_4 35 1

In [100]:
!fslmerge -t $B0_all $B0_1 $B0_2 $B0_3 $B0_4
!fslmaths $B0_all -Tmean $B0
!fslmerge -t $DATA $DATA_s $B0_all

In [101]:
def run_diff2standard(b0, t1, t1_brain, t1_2_mni_warp, mni_ref,
                      diff2str_prefix, diff2std_warp):
    """
    Build a diffusion -> MNI warp.

    1) epi_reg:     b0 -> T1 (BBR), writes <prefix>.mat
    2) convertwarp: combine <prefix>.mat with the T1 -> MNI FNIRT warp

    Returns: path to the diff2standard warp field.
    """
    Path(diff2str_prefix).parent.mkdir(parents=True, exist_ok=True)

    # --- Step 1: diffusion -> T1 ---
    epi_cmd = [
        "epi_reg",
        f"--epi={b0}",
        f"--t1={t1}",
        f"--t1brain={t1_brain}",
        f"--out={diff2str_prefix}",
    ]
    _run(epi_cmd)

    # --- Step 2: concatenate with T1 -> MNI warp ---
    _run([
        "convertwarp",
        f"--ref={mni_ref}",
        f"--premat={diff2str_prefix}.mat",
        f"--warp1={t1_2_mni_warp}",
        f"--out={diff2std_warp}",
    ])
    return diff2std_warp

In [102]:
def apply_diff2standard(img, mni_ref, diff2std_warp, out, interp="trilinear"):
    """Move b0 into MNI space."""
    _run([
        "applywarp",
        f"--in={img}",
        f"--ref={mni_ref}",
        f"--warp={diff2std_warp}",
        f"--out={out}",
        f"--interp={interp}",
    ])
    return out

if __name__ == "__main__":
    run_diff2standard(B0, T1, T1_brain, COUT, MNI_HEAD,
                      DIFF2STR, DIFF2STD)

epi_reg --epi=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/dwi/b0_av.nii.gz --t1=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/input/raw/sub-15/anat/sub-15_ses-01_T1w.nii.gz --t1brain=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/anat/sub-15_ses-01_T1w_brain.nii.gz --out=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/xfms/diff2str
Running FAST segmentation
FLIRT pre-alignment
Running BBR
0.229697 0.999995 -0.002447 0.002152 0.000000 0.002456 0.999988 -0.004201 0.000000 -0.002142 0.004206 0.999989 0.000000 0.215591 -0.442135 1.471751 1.000000 
convertwarp --ref=/Users/ecd25/fsl/data/standard/MNI152_T1_2mm.nii.gz --premat=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/xfms/diff2str.mat --warp1=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/xfms/sub-15_T1_2_MNI_warp.nii.gz --out=/Users/ecd25/Desktop/BENCH/BENCH_Tuto

In [103]:
apply_diff2standard(B0, MNI_HEAD, DIFF2STD, B0_in_MNI)

applywarp --in=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/dwi/b0_av.nii.gz --ref=/Users/ecd25/fsl/data/standard/MNI152_T1_2mm.nii.gz --warp=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/xfms/diff2standard_warp.nii.gz --out=/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/xfms/sub-15_b0_in_MNI.nii.gz --interp=trilinear


'/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/output/derivatives/sub-15/xfms/sub-15_b0_in_MNI.nii.gz'

In [106]:
!bench diff-summary --data $DATA --bvecs $bvecs --bvals $bvals --mask $WM_in_MNI --xfm $DIFF2STD --summarytype dt --output $SUM

Summary measurements are computed.


In [108]:
SUM_all = "/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/summaries"
D_mat = "/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/design.mat"
D_con = "/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/design.con"
subjlist = "/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/subjlist.txt"
GLM = "/Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/GLM"

In [109]:
!bench glm --summarydir $SUM_all \
          --mask $WM_in_MNI \
          --designmat $D_mat \
          --designcon $D_con \
          --subjectlist $subjlist \
          --output $GLM

loaded summaries from 3 subjects.
running glm for 3 subjects
GLM is done. Results are stored in /Users/ecd25/Desktop/BENCH/BENCH_Tutorial/one_vs_many/GLM
